## Example 1: Basic urban morphology analysis with building footprints

In [ ]:
import momepy
import geopandas as gpd
import pandas as pd
from shapely.geometry import Polygon, Point, LineString
import numpy as np

# Example 1: Basic urban morphology analysis with building footprints
print("momepy basic urban morphology analysis")

try:
    print("Example 1 - Basic urban morphology analysis with building footprints")
    
    # Create sample building footprints for analysis
    print("Creating sample building footprint data:")
    
    # Generate synthetic building polygons
    buildings_data = []
    building_id = 0
    
    # Create a grid of building footprints with variations
    for i in range(5):
        for j in range(4):
            # Base coordinates
            x_base = j * 50 + np.random.uniform(-5, 5)
            y_base = i * 50 + np.random.uniform(-5, 5)
            
            # Random building dimensions
            width = np.random.uniform(15, 25)
            height = np.random.uniform(12, 30)
            
            # Create rectangular building
            building = Polygon([
                (x_base, y_base),
                (x_base + width, y_base),
                (x_base + width, y_base + height),
                (x_base, y_base + height)
            ])
            
            buildings_data.append({
                'geometry': building,
                'building_id': building_id,
                'type': np.random.choice(['residential', 'commercial', 'mixed']),
                'height': np.random.uniform(3, 15)  # Building height in meters
            })
            building_id += 1
    
    # Create GeoDataFrame
    buildings_gdf = gpd.GeoDataFrame(buildings_data, crs='EPSG:3857')
    
    print(f"  Created {len(buildings_gdf)} building footprints")
    print(f"  Building types: {buildings_gdf['type'].value_counts().to_dict()}")
    print(f"  Average building area: {buildings_gdf.geometry.area.mean():.2f} m²")
    print(f"  Total built area: {buildings_gdf.geometry.area.sum():.2f} m²")
    
    # Basic morphometric measurements
    print(f"\nCalculating basic morphometric measurements:")
    
    # Area measurements
    # momepy measures are plain functions that take a GeoDataFrame and return a pandas Series;
    # area and perimeter come straight from geopandas (`.area`, `.length`).
    buildings_gdf['area'] = buildings_gdf.geometry.area
    print(f"  Area calculation: {len(buildings_gdf)} buildings processed")
    print(f"  Min area: {buildings_gdf['area'].min():.2f} m²")
    print(f"  Max area: {buildings_gdf['area'].max():.2f} m²")
    print(f"  Mean area: {buildings_gdf['area'].mean():.2f} m²")
    
    # Perimeter measurements
    buildings_gdf['perimeter'] = buildings_gdf.geometry.length
    print(f"  Perimeter calculation: {len(buildings_gdf)} buildings processed")
    print(f"  Average perimeter: {buildings_gdf['perimeter'].mean():.2f} m")
    
    # Compactness measurements
    buildings_gdf['circular_compactness'] = momepy.circular_compactness(buildings_gdf)
    print(f"  Circular compactness: mean = {buildings_gdf['circular_compactness'].mean():.4f}")
    
    # Equivalent rectangular index
    buildings_gdf['eri'] = momepy.equivalent_rectangular_index(buildings_gdf)
    print(f"  Equivalent Rectangular Index: mean = {buildings_gdf['eri'].mean():.4f}")
    
    # Elongation measurements
    buildings_gdf['elongation'] = momepy.elongation(buildings_gdf)
    print(f"  Elongation: mean = {buildings_gdf['elongation'].mean():.4f}")
    
    # Convexity measurements
    buildings_gdf['convexity'] = momepy.convexity(buildings_gdf)
    print(f"  Convexity: mean = {buildings_gdf['convexity'].mean():.4f}")
    # Every footprint is a rectangle: convex, its own equivalent rectangle, and at most as
    # compact as a square (2/pi of a circle's compactness)
    assert len(buildings_gdf) == 20
    assert np.allclose(buildings_gdf['convexity'], 1) and np.allclose(buildings_gdf['eri'], 1)
    assert ((buildings_gdf['circular_compactness'] > 0) & (buildings_gdf['circular_compactness'] <= 2 / np.pi + 1e-9)).all()
    assert ((buildings_gdf['elongation'] > 0) & (buildings_gdf['elongation'] <= 1)).all()
    
    # Shape analysis
    print(f"\nShape analysis results:")
    
    # Find most compact buildings
    most_compact = buildings_gdf.nlargest(3, 'circular_compactness')[['building_id', 'circular_compactness', 'area']]
    print(f"  Most compact buildings:")
    for _, building in most_compact.iterrows():
        print(f"    Building {building['building_id']}: compactness = {building['circular_compactness']:.4f}, area = {building['area']:.2f} m²")
    
    # Find most elongated buildings
    most_elongated = buildings_gdf.nlargest(3, 'elongation')[['building_id', 'elongation', 'area']]
    print(f"  Most elongated buildings:")
    for _, building in most_elongated.iterrows():
        print(f"    Building {building['building_id']}: elongation = {building['elongation']:.4f}, area = {building['area']:.2f} m²")
    
    # Area distribution analysis
    print(f"\nArea distribution analysis:")
    area_stats = buildings_gdf['area'].describe()
    print(f"  Q1 (25th percentile): {area_stats['25%']:.2f} m²")
    print(f"  Median: {area_stats['50%']:.2f} m²")
    print(f"  Q3 (75th percentile): {area_stats['75%']:.2f} m²")
    print(f"  Standard deviation: {area_stats['std']:.2f} m²")
    
    # Building type analysis
    print(f"\nBuilding type morphology:")
    for building_type in buildings_gdf['type'].unique():
        type_buildings = buildings_gdf[buildings_gdf['type'] == building_type]
        print(f"  {building_type.title()} buildings ({len(type_buildings)}):")
        print(f"    Average area: {type_buildings['area'].mean():.2f} m²")
        print(f"    Average compactness: {type_buildings['circular_compactness'].mean():.4f}")
        print(f"    Average elongation: {type_buildings['elongation'].mean():.4f}")
    
    # Basic clustering of building characteristics
    print(f"\nBuilding characteristic clusters:")
    
    # Simple clustering based on area and compactness
    large_compact = buildings_gdf[
        (buildings_gdf['area'] > buildings_gdf['area'].median()) & 
        (buildings_gdf['circular_compactness'] > buildings_gdf['circular_compactness'].median())
    ]
    
    small_elongated = buildings_gdf[
        (buildings_gdf['area'] < buildings_gdf['area'].median()) & 
        (buildings_gdf['elongation'] > buildings_gdf['elongation'].median())
    ]
    
    print(f"  Large & compact buildings: {len(large_compact)} ({len(large_compact)/len(buildings_gdf)*100:.1f}%)")
    print(f"  Small & elongated buildings: {len(small_elongated)} ({len(small_elongated)/len(buildings_gdf)*100:.1f}%)")
    
    # Summary statistics table
    print(f"\nMorphometric summary:")
    morphometric_cols = ['area', 'perimeter', 'circular_compactness', 'elongation', 'convexity', 'eri']
    summary_stats = buildings_gdf[morphometric_cols].agg(['count', 'mean', 'std', 'min', 'max'])
    
    for metric in morphometric_cols[:4]:  # Show first 4 metrics
        stats = summary_stats[metric]
        print(f"  {metric}:")
        print(f"    Mean: {stats['mean']:.4f}, Std: {stats['std']:.4f}")
        print(f"    Range: {stats['min']:.4f} - {stats['max']:.4f}")
    
    print("momepy example 1 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"momepy example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 1 finished")

## Example 2: Tessellation and spatial units analysis

In [ ]:
import momepy
import geopandas as gpd
import pandas as pd
from shapely.geometry import Polygon, Point, LineString, MultiPolygon
import numpy as np

# Example 2: Tessellation and spatial units analysis
print("momepy tessellation and spatial units analysis")

try:
    print("Example 2 - Tessellation and spatial units analysis")
    
    # Create building footprints for tessellation
    print("Creating buildings for tessellation analysis:")
    
    buildings_data = []
    for i in range(6):
        for j in range(5):
            x = j * 60 + np.random.uniform(-10, 10)
            y = i * 45 + np.random.uniform(-8, 8)
            
            # Rectangular buildings only for stability
            w, h = np.random.uniform(12, 25), np.random.uniform(10, 20)
            coords = [(x, y), (x + w, y), (x + w, y + h), (x, y + h), (x, y)]  # Ensure closed
            
            buildings_data.append({
                'geometry': Polygon(coords),
                'uID': i * 5 + j,
                'type': np.random.choice(['residential', 'commercial', 'office'])
            })
    
    buildings_gdf = gpd.GeoDataFrame(buildings_data, crs='EPSG:3857')
    print(f"  Created {len(buildings_gdf)} buildings for tessellation")
    
    # Generate morphological tessellation using new functional API
    print(f"\nGenerating morphological tessellation:")
    
    # Use new functional API
    tessellation_gdf = momepy.morphological_tessellation(buildings_gdf, shrink=0.4)
    
    # Join building attributes back using index
    tessellation_gdf = tessellation_gdf.join(buildings_gdf[['uID', 'type']], how='left')
    
    print(f"  Generated {len(tessellation_gdf)} tessellation cells")
    print(f"  Average cell area: {tessellation_gdf.geometry.area.mean():.2f} m²")
    print(f"  Total tessellation area: {tessellation_gdf.geometry.area.sum():.2f} m²")
    
    # Calculate tessellation-based metrics
    print(f"\nCalculating tessellation-based metrics:")
    
    # Area of tessellation cells (a geopandas property)
    tessellation_gdf['tess_area'] = tessellation_gdf.geometry.area
    print(f"  Tessellation areas calculated")
    print(f"  Min cell area: {tessellation_gdf['tess_area'].min():.2f} m²")
    print(f"  Max cell area: {tessellation_gdf['tess_area'].max():.2f} m²")
    print(f"  Cell area standard deviation: {tessellation_gdf['tess_area'].std():.2f} m²")
    
    # Compactness of tessellation cells
    tessellation_gdf['tess_compactness'] = momepy.circular_compactness(tessellation_gdf)
    print(f"  Tessellation compactness: mean = {tessellation_gdf['tess_compactness'].mean():.4f}")
    
    # Orientation of tessellation cells
    tessellation_gdf['orientation'] = momepy.orientation(tessellation_gdf)
    print(f"  Tessellation orientation: mean = {tessellation_gdf['orientation'].mean():.2f}°")
    
    # Coverage analysis
    print(f"\nCoverage analysis:")
    
    # Building coverage within tessellation cells
    buildings_gdf['building_area'] = buildings_gdf.geometry.area
    
    # Merge building data with tessellation
    merged_gdf = tessellation_gdf.merge(
        buildings_gdf[['uID', 'building_area']], 
        on='uID', 
        how='left',
        suffixes=('', '_building')
    )
    
    merged_gdf['coverage_ratio'] = merged_gdf['building_area'] / merged_gdf['tess_area']
    
    print(f"  Coverage ratios calculated for {len(merged_gdf)} cells")
    print(f"  Mean coverage ratio: {merged_gdf['coverage_ratio'].mean():.4f}")
    print(f"  Max coverage ratio: {merged_gdf['coverage_ratio'].max():.4f}")
    print(f"  Min coverage ratio: {merged_gdf['coverage_ratio'].min():.4f}")
    # One cell per building, and each cell contains its building
    assert len(tessellation_gdf) == len(buildings_gdf) == 30
    assert ((merged_gdf['coverage_ratio'] > 0) & (merged_gdf['coverage_ratio'] <= 1 + 1e-9)).all()
    
    # Coverage by building type
    print(f"\nCoverage by building type:")
    for building_type in merged_gdf['type'].dropna().unique():
        type_cells = merged_gdf[merged_gdf['type'] == building_type]
        print(f"  {building_type.title()}: mean coverage = {type_cells['coverage_ratio'].mean():.4f}")
    
    # Tessellation quality metrics
    print(f"\nTessellation quality metrics:")
    
    # Calculate area distribution uniformity
    area_cv = tessellation_gdf['tess_area'].std() / tessellation_gdf['tess_area'].mean()
    print(f"  Area coefficient of variation: {area_cv:.4f}")
    
    # Compactness distribution
    compact_cells = (tessellation_gdf['tess_compactness'] > 0.7).sum()
    print(f"  Highly compact cells (>0.7): {compact_cells} ({compact_cells/len(tessellation_gdf)*100:.1f}%)")
    
    # Density analysis
    print(f"\nDensity analysis:")
    
    # Calculate building density per tessellation cell
    density_data = []
    for idx, tess_cell in tessellation_gdf.iterrows():
        tess_geom = tess_cell.geometry
        intersecting_buildings = buildings_gdf[buildings_gdf.geometry.intersects(tess_geom)]
        
        building_count = len(intersecting_buildings)
        building_density = building_count / (tess_cell['tess_area'] / 10000)  # buildings per hectare
        
        density_data.append({
            'tess_id': idx,
            'building_count': building_count,
            'density_per_ha': building_density,
            'tess_area': tess_cell['tess_area']
        })
    
    density_df = pd.DataFrame(density_data)
    
    print(f"  Cells analyzed: {len(density_df)}")
    print(f"  Average building density: {density_df['density_per_ha'].mean():.2f} buildings/ha")
    print(f"  Max building density: {density_df['density_per_ha'].max():.2f} buildings/ha")
    
    # Identify dense vs sparse areas
    high_density_threshold = density_df['density_per_ha'].quantile(0.75)
    high_density_cells = density_df[density_df['density_per_ha'] >= high_density_threshold]
    
    print(f"  High-density cells (>{high_density_threshold:.1f}/ha): {len(high_density_cells)} cells")
    if len(high_density_cells) > 0:
        print(f"  Average density in high-density cells: {high_density_cells['density_per_ha'].mean():.2f} buildings/ha")
    
    # Tessellation pattern analysis
    print(f"\nTessellation pattern analysis:")
    
    # Regularity metrics
    orientation_variation = tessellation_gdf['orientation'].std()
    print(f"  Orientation variation (std): {orientation_variation:.2f}°")
    
    area_uniformity = 1 - (tessellation_gdf['tess_area'].std() / tessellation_gdf['tess_area'].mean())
    print(f"  Area uniformity index: {area_uniformity:.4f} (1.0 = perfectly uniform)")
    
    # Shape complexity
    avg_compactness = tessellation_gdf['tess_compactness'].mean()
    print(f"  Average shape compactness: {avg_compactness:.4f} (1.0 = perfect circle)")
    
    # Spatial distribution metrics
    print(f"\nSpatial distribution metrics:")
    
    # Calculate centroids for distance analysis
    buildings_gdf['centroid'] = buildings_gdf.geometry.centroid
    
    # Nearest neighbor distances between buildings
    building_centroids = [(geom.x, geom.y) for geom in buildings_gdf['centroid']]
    
    nearest_distances = []
    for i, (x1, y1) in enumerate(building_centroids):
        distances = [np.sqrt((x1-x2)**2 + (y1-y2)**2) for j, (x2, y2) in enumerate(building_centroids) if i != j]
        if distances:
            nearest_distances.append(min(distances))
    
    print(f"  Average nearest neighbor distance: {np.mean(nearest_distances):.2f} m")
    print(f"  Nearest neighbor std: {np.std(nearest_distances):.2f} m")
    
    # Tessellation applications summary
    print(f"\nTessellation applications:")
    applications = [
        "Urban density analysis: Measure building density per spatial unit",
        "Coverage assessment: Calculate built-up area ratios",
        "Morphological characterization: Analyze urban form patterns",
        "Comparative studies: Compare different urban areas objectively",
        "Planning support: Identify development opportunities"
    ]
    
    for app in applications:
        print(f"  • {app}")
    
    print("momepy example 2 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    import traceback
    traceback.print_exc()
    raise
    
print("Example 2 finished")

## Example 3: Street network analysis and connectivity

In [ ]:
import momepy
import geopandas as gpd
import pandas as pd
from shapely.geometry import Polygon, Point, LineString
import numpy as np

# Example 3: Street network analysis and connectivity
print("momepy street network analysis and connectivity")

try:
    print("Example 3 - Street network analysis and connectivity")
    
    # Create sample street network
    print("Creating sample street network:")
    
    # Generate a grid-like street network with some variations
    streets_data = []
    street_id = 0
    
    # Create main horizontal streets
    for i in range(4):
        y = i * 80
        # Simple segments without much curvature
        segments = [
            LineString([(0, y), (100, y)]),
            LineString([(100, y), (200, y)]),
            LineString([(200, y), (300, y)])
        ]
        
        for segment in segments:
            streets_data.append({
                'geometry': segment,
                'street_id': street_id,
                'street_type': 'main',
                'width': np.random.uniform(8, 12)
            })
            street_id += 1
    
    # Create vertical connecting streets
    for j in range(5):
        x = j * 75
        segments = [
            LineString([(x, 0), (x, 80)]),
            LineString([(x, 80), (x, 160)]),
            LineString([(x, 160), (x, 240)])
        ]
        
        for segment in segments:
            streets_data.append({
                'geometry': segment,
                'street_id': street_id,
                'street_type': 'secondary',
                'width': np.random.uniform(6, 10)
            })
            street_id += 1
    
    # Add some diagonal connections
    diagonals = [
        LineString([(50, 40), (150, 120)]),
        LineString([(150, 40), (250, 120)]),
        LineString([(100, 120), (200, 200)])
    ]
    
    for diagonal in diagonals:
        streets_data.append({
            'geometry': diagonal,
            'street_id': street_id,
            'street_type': 'connector',
            'width': np.random.uniform(5, 8)
        })
        street_id += 1
    
    # Create GeoDataFrame
    streets_gdf = gpd.GeoDataFrame(streets_data, crs='EPSG:3857')
    
    print(f"  Created street network with {len(streets_gdf)} segments")
    print(f"  Street types: {streets_gdf['street_type'].value_counts().to_dict()}")
    print(f"  Total network length: {streets_gdf.geometry.length.sum():.2f} m")
    print(f"  Average street width: {streets_gdf['width'].mean():.2f} m")
    
    # Basic network metrics
    print(f"\nCalculating basic network metrics:")
    
    # Street length analysis (using .length attribute directly)
    streets_gdf['length'] = streets_gdf.geometry.length
    print(f"  Street lengths calculated")
    print(f"  Average segment length: {streets_gdf['length'].mean():.2f} m")
    print(f"  Longest segment: {streets_gdf['length'].max():.2f} m")
    print(f"  Shortest segment: {streets_gdf['length'].min():.2f} m")
    # 12 horizontal and 15 vertical segments plus 3 diagonals of 100 x 80 m
    assert len(streets_gdf) == 30
    assert abs(streets_gdf['length'].sum() - (1200 + 1200 + 3 * np.hypot(100, 80))) < 1e-6

    # momepy turns the street GeoDataFrame into a networkx graph (one edge per segment)
    street_graph = momepy.gdf_to_nx(streets_gdf, approach='primal')
    print(f"  momepy graph: {street_graph.number_of_nodes()} nodes, {street_graph.number_of_edges()} edges")
    assert street_graph.number_of_edges() == len(streets_gdf)
    
    # Network connectivity analysis
    print(f"\nNetwork connectivity analysis:")
    
    # Extract endpoints of all street segments
    endpoints = []
    segment_endpoints = {}
    
    for idx, street in streets_gdf.iterrows():
        line = street.geometry
        start_point = Point(line.coords[0])
        end_point = Point(line.coords[-1])
        
        endpoints.extend([start_point, end_point])
        segment_endpoints[street['street_id']] = (start_point, end_point)
    
    # Find intersections (simplified - points that are close together)
    intersections = []
    intersection_tolerance = 5.0  # meters
    
    processed_points = set()
    for i, point1 in enumerate(endpoints):
        if i in processed_points:
            continue
            
        cluster_points = [point1]
        cluster_indices = [i]
        
        for j, point2 in enumerate(endpoints[i+1:], i+1):
            if j in processed_points:
                continue
                
            if point1.distance(point2) <= intersection_tolerance:
                cluster_points.append(point2)
                cluster_indices.append(j)
        
        if len(cluster_points) > 1:  # Real intersection
            # Calculate centroid of cluster
            x_coords = [p.x for p in cluster_points]
            y_coords = [p.y for p in cluster_points]
            centroid = Point(np.mean(x_coords), np.mean(y_coords))
            
            intersections.append({
                'geometry': centroid,
                'intersection_id': len(intersections),
                'degree': len([seg_id for seg_id, (start, end) in segment_endpoints.items() 
                              if any(p.distance(centroid) <= intersection_tolerance 
                                   for p in [start, end])])
            })
            
            processed_points.update(cluster_indices)
    
    intersections_gdf = gpd.GeoDataFrame(intersections, crs=streets_gdf.crs)
    
    print(f"  Found {len(intersections_gdf)} intersections")
    print(f"  Average intersection degree: {intersections_gdf['degree'].mean():.2f}")
    print(f"  Max intersection degree: {intersections_gdf['degree'].max()}")
    assert len(intersections_gdf) > 0 and (intersections_gdf['degree'] >= 2).all()
    
    # Network density metrics
    print(f"\nNetwork density metrics:")
    
    # Calculate network area (bounding box)
    network_bounds = streets_gdf.total_bounds
    network_area = (network_bounds[2] - network_bounds[0]) * (network_bounds[3] - network_bounds[1])
    
    # Street density
    street_density = streets_gdf['length'].sum() / (network_area / 1000000)  # km/km²
    print(f"  Street density: {street_density:.2f} km/km²")
    
    # Intersection density
    intersection_density = len(intersections_gdf) / (network_area / 1000000)  # intersections/km²
    print(f"  Intersection density: {intersection_density:.2f} intersections/km²")
    
    # Connectivity index (intersections per km of street)
    connectivity_index = len(intersections_gdf) / (streets_gdf['length'].sum() / 1000)
    print(f"  Connectivity index: {connectivity_index:.2f} intersections/km")
    
    # Street network characterization
    print(f"\nStreet network characterization:")
    
    # Analyze intersection types
    intersection_types = intersections_gdf['degree'].value_counts().sort_index()
    print(f"  Intersection degree distribution:")
    for degree, count in intersection_types.items():
        intersection_type = {
            2: "Dead end",
            3: "T-junction", 
            4: "Cross-roads",
            5: "5-way junction"
        }.get(degree, f"{degree}-way junction")
        
        print(f"    {intersection_type}: {count} ({count/len(intersections_gdf)*100:.1f}%)")
    
    # Network efficiency metrics
    print(f"\nNetwork efficiency metrics:")
    
    # Average segment length by street type
    for street_type in streets_gdf['street_type'].unique():
        type_streets = streets_gdf[streets_gdf['street_type'] == street_type]
        print(f"  {street_type.title()} streets:")
        print(f"    Count: {len(type_streets)}")
        print(f"    Average length: {type_streets['length'].mean():.2f} m")
        print(f"    Average width: {type_streets['width'].mean():.2f} m")
    
    # Street network quality indicators
    print(f"\nStreet network quality indicators:")
    
    quality_indicators = [
        f"Connectivity: {connectivity_index:.2f} intersections/km ({'Good' if connectivity_index > 2 else 'Moderate'})",
        f"Density: {street_density:.2f} km/km² ({'Dense' if street_density > 10 else 'Moderate'})",
        f"Intersection variety: {len(intersection_types)} different junction types",
        f"Network redundancy: {(intersection_types.get(4, 0) + intersection_types.get(5, 0))} cross-roads and complex junctions"
    ]
    
    for indicator in quality_indicators:
        print(f"  • {indicator}")
    
    # Network analysis applications
    print(f"\nNetwork analysis applications:")
    applications = [
        "Transportation planning: Optimize traffic flow and accessibility",
        "Urban design: Evaluate walkability and connectivity",
        "Emergency services: Assess response time and route redundancy",
        "Land use planning: Understand development potential",
        "Infrastructure planning: Identify critical nodes"
    ]
    
    for app in applications:
        print(f"  • {app}")
    
    print("momepy example 3 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    import traceback
    traceback.print_exc()
    raise
    
print("Example 3 finished")

## Example 4: Urban diversity and land use analysis

In [ ]:
import momepy
import geopandas as gpd
import pandas as pd
from shapely.geometry import Polygon, Point, LineString
import numpy as np

# Example 4: Urban diversity and land use analysis  
print("momepy urban diversity and land use analysis")

try:
    print("Example 4 - Urban diversity and land use analysis")
    
    # Create comprehensive urban dataset
    print("Creating comprehensive urban dataset:")
    
    # Generate buildings with detailed attributes
    buildings_data = []
    building_id = 0
    
    # Create different urban zones
    zones = {
        'residential': {'x_range': (0, 150), 'y_range': (0, 200), 'density': 'high'},
        'commercial': {'x_range': (150, 300), 'y_range': (0, 100), 'density': 'medium'},
        'mixed_use': {'x_range': (150, 300), 'y_range': (100, 200), 'density': 'medium'},
        'industrial': {'x_range': (0, 150), 'y_range': (200, 300), 'density': 'low'}
    }
    
    for zone_type, zone_config in zones.items():
        x_min, x_max = zone_config['x_range']
        y_min, y_max = zone_config['y_range']
        density = zone_config['density']
        
        # Adjust building density based on zone
        density_factor = {'high': 1.0, 'medium': 0.7, 'low': 0.4}[density]
        num_buildings = int(20 * density_factor)
        
        for _ in range(num_buildings):
            x = np.random.uniform(x_min, x_max)
            y = np.random.uniform(y_min, y_max)
            
            # Zone-specific building characteristics
            if zone_type == 'residential':
                width = np.random.uniform(8, 18)
                height = np.random.uniform(10, 25)
                floors = np.random.randint(1, 4)
                age = np.random.choice(['new', 'medium', 'old'], p=[0.3, 0.4, 0.3])
                
            elif zone_type == 'commercial':
                width = np.random.uniform(15, 40)
                height = np.random.uniform(12, 35)
                floors = np.random.randint(1, 5)
                age = np.random.choice(['new', 'medium', 'old'], p=[0.5, 0.3, 0.2])
                
            elif zone_type == 'mixed_use':
                width = np.random.uniform(12, 30)
                height = np.random.uniform(15, 45)
                floors = np.random.randint(2, 8)
                age = np.random.choice(['new', 'medium', 'old'], p=[0.4, 0.4, 0.2])
                
            else:  # industrial
                width = np.random.uniform(25, 80)
                height = np.random.uniform(20, 60)
                floors = np.random.randint(1, 3)
                age = np.random.choice(['new', 'medium', 'old'], p=[0.2, 0.3, 0.5])
            
            # Create building polygon (ensuring it's closed)
            building = Polygon([
                (x, y), (x + width, y), 
                (x + width, y + height), (x, y + height), (x, y)
            ])
            
            buildings_data.append({
                'geometry': building,
                'building_id': building_id,
                'zone_type': zone_type,
                'floors': floors,
                'age': age,
                'height_m': floors * 3.5,  # Approximate height
                'use_intensity': np.random.uniform(0.3, 1.0)
            })
            building_id += 1
    
    buildings_gdf = gpd.GeoDataFrame(buildings_data, crs='EPSG:3857')
    
    print(f"  Created {len(buildings_gdf)} buildings across {len(zones)} zones")
    print(f"  Zone distribution: {buildings_gdf['zone_type'].value_counts().to_dict()}")
    print(f"  Age distribution: {buildings_gdf['age'].value_counts().to_dict()}")
    print(f"  Floor range: {buildings_gdf['floors'].min()}-{buildings_gdf['floors'].max()} floors")
    
    # Diversity metrics calculation
    print(f"\nCalculating diversity metrics:")
    
    # Functional diversity analysis
    print(f"\nFunctional diversity analysis:")
    
    # Simpson's diversity index for land use
    zone_counts = buildings_gdf['zone_type'].value_counts()
    total_buildings = len(buildings_gdf)
    simpson_index = 1 - sum((count/total_buildings)**2 for count in zone_counts)
    
    print(f"  Simpson's diversity index: {simpson_index:.4f} (0=no diversity, 1=max diversity)")
    print(f"  Functional categories: {len(zone_counts)} different zone types")
    
    # Shannon diversity index
    shannon_index = -sum((count/total_buildings) * np.log(count/total_buildings) for count in zone_counts)
    print(f"  Shannon diversity index: {shannon_index:.4f}")
    # 20 + 14 + 14 + 8 buildings across four zones
    assert len(buildings_gdf) == 56 and set(zone_counts.index) == set(zones)
    assert 0 < simpson_index < 1 and 0 < shannon_index <= np.log(len(zones)) + 1e-9
    
    # Detailed zone analysis
    print(f"\nDetailed zone analysis:")
    for zone in zone_counts.index:
        zone_buildings = buildings_gdf[buildings_gdf['zone_type'] == zone]
        zone_area = zone_buildings.geometry.area.sum()
        avg_intensity = zone_buildings['use_intensity'].mean()
        
        print(f"  {zone.title()} zone:")
        print(f"    Buildings: {len(zone_buildings)} ({len(zone_buildings)/total_buildings*100:.1f}%)")
        print(f"    Total floor area: {zone_area:.0f} m²")
        print(f"    Average use intensity: {avg_intensity:.3f}")
    
    # Morphological diversity
    print(f"\nMorphological diversity:")
    
    # Calculate morphological metrics using functional API
    buildings_gdf['area'] = buildings_gdf.geometry.area
    buildings_gdf['perimeter'] = buildings_gdf.geometry.length
    buildings_gdf['compactness'] = momepy.circular_compactness(buildings_gdf)
    buildings_gdf['elongation'] = momepy.elongation(buildings_gdf)
    assert ((buildings_gdf['compactness'] > 0) & (buildings_gdf['compactness'] <= 2 / np.pi + 1e-9)).all()
    assert ((buildings_gdf['elongation'] > 0) & (buildings_gdf['elongation'] <= 1)).all()
    
    # Morphological diversity by zone
    morph_metrics = ['area', 'compactness', 'elongation', 'height_m']
    
    for zone in buildings_gdf['zone_type'].unique():
        zone_buildings = buildings_gdf[buildings_gdf['zone_type'] == zone]
        print(f"  {zone.title()} morphological characteristics:")
        
        for metric in morph_metrics:
            values = zone_buildings[metric]
            cv = values.std() / values.mean()  # Coefficient of variation
            print(f"    {metric}: mean={values.mean():.2f}, CV={cv:.3f}")
    
    # Spatial diversity analysis
    print(f"\nSpatial diversity analysis:")
    
    # Local diversity (neighborhood scale)
    neighborhood_size = 100  # meters
    
    diversity_scores = []
    for _, building in buildings_gdf.iterrows():
        building_point = building.geometry.centroid
        
        # Create buffer around building
        buffer = building_point.buffer(neighborhood_size)
        
        # Find neighboring buildings
        neighbors = buildings_gdf[buildings_gdf.geometry.intersects(buffer)]
        
        if len(neighbors) > 1:
            # Calculate local diversity
            neighbor_zones = neighbors['zone_type'].value_counts()
            local_simpson = 1 - sum((count/len(neighbors))**2 for count in neighbor_zones)
        else:
            local_simpson = 0
        
        diversity_scores.append({
            'building_id': building['building_id'],
            'local_diversity': local_simpson,
            'neighbor_count': len(neighbors) - 1,
            'zone_type': building['zone_type']
        })
    
    diversity_df = pd.DataFrame(diversity_scores)
    
    print(f"  Local diversity analysis completed")
    print(f"  Average local diversity: {diversity_df['local_diversity'].mean():.4f}")
    print(f"  Average neighborhood size: {diversity_df['neighbor_count'].mean():.1f} buildings")
    
    # Diversity by zone
    for zone in buildings_gdf['zone_type'].unique():
        zone_diversity = diversity_df[diversity_df['zone_type'] == zone]['local_diversity'].mean()
        print(f"  Average local diversity in {zone}: {zone_diversity:.4f}")
    
    # Temporal diversity (age-based)
    print(f"\nTemporal diversity analysis:")
    
    age_counts = buildings_gdf['age'].value_counts()
    age_simpson = 1 - sum((count/total_buildings)**2 for count in age_counts)
    
    print(f"  Age diversity (Simpson): {age_simpson:.4f}")
    print(f"  Age distribution:")
    for age, count in age_counts.items():
        print(f"    {age.title()}: {count} buildings ({count/total_buildings*100:.1f}%)")
    
    # Mixed-use analysis
    print(f"\nMixed-use analysis:")
    
    # Identify areas with high functional diversity
    high_diversity_buildings = diversity_df[diversity_df['local_diversity'] > diversity_df['local_diversity'].quantile(0.75)]
    
    print(f"  High-diversity areas: {len(high_diversity_buildings)} buildings")
    print(f"  Mixed-use zones: {high_diversity_buildings['zone_type'].value_counts().to_dict()}")
    
    # Intensity analysis
    print(f"\nUse intensity analysis:")
    
    # Floor Area Ratio (FAR) approximation
    buildings_gdf['footprint_area'] = buildings_gdf['area']
    buildings_gdf['total_floor_area'] = buildings_gdf['footprint_area'] * buildings_gdf['floors']
    
    # Calculate FAR by zone (simplified)
    for zone in buildings_gdf['zone_type'].unique():
        zone_buildings = buildings_gdf[buildings_gdf['zone_type'] == zone]
        total_footprint = zone_buildings['footprint_area'].sum()
        total_floor_area = zone_buildings['total_floor_area'].sum()
        
        # Approximate FAR
        far = total_floor_area / total_footprint if total_footprint > 0 else 0
        
        print(f"  {zone.title()} zone:")
        print(f"    Approximate FAR: {far:.2f}")
        print(f"    Average use intensity: {zone_buildings['use_intensity'].mean():.3f}")
    
    # Diversity analysis summary
    print(f"\nDiversity analysis summary:")
    summary_metrics = [
        f"Overall functional diversity (Simpson): {simpson_index:.4f}",
        f"Mixed-use prevalence: {len(high_diversity_buildings)/len(buildings_gdf)*100:.1f}% of buildings",
        f"Age diversity: {age_simpson:.4f} (temporal variation)",
        f"Morphological variety: {len(buildings_gdf['zone_type'].unique())} distinct building types"
    ]
    
    for metric in summary_metrics:
        print(f"  • {metric}")
    
    print("momepy example 4 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    import traceback
    traceback.print_exc()
    raise
    
print("Example 4 finished")

## Example 5: Package capabilities and morphology metrics

In [ ]:
import momepy
import geopandas as gpd
import pandas as pd
from shapely.geometry import Polygon, Point
import numpy as np

# Example 5: Package capabilities and morphology metrics
print("momepy package capabilities and morphology metrics")

print("Example 5 - Package capabilities and morphology metrics")

# Package overview
print(f"\nmomepy Package Overview:")
if hasattr(momepy, '__version__'):
    print(f"  Version: {momepy.__version__}")

# Create test buildings
print(f"\nCreating test dataset:")
buildings_data = []
for i in range(20):
    x, y = np.random.uniform(0, 200, 2)
    w, h = np.random.uniform(10, 30, 2)
    building = Polygon([(x, y), (x+w, y), (x+w, y+h), (x, y+h), (x, y)])
    buildings_data.append({'geometry': building, 'uID': i})

buildings_gdf = gpd.GeoDataFrame(buildings_data, crs='EPSG:3857')
print(f"  Created {len(buildings_gdf)} buildings")

# Demonstrate functional API (new)
print(f"\nFunctional API (new, recommended):")

# Use functional API functions
buildings_gdf['area'] = buildings_gdf.geometry.area
buildings_gdf['perimeter'] = buildings_gdf.geometry.length
buildings_gdf['compactness'] = momepy.circular_compactness(buildings_gdf)
buildings_gdf['elongation'] = momepy.elongation(buildings_gdf)
buildings_gdf['convexity'] = momepy.convexity(buildings_gdf)
buildings_gdf['orientation'] = momepy.orientation(buildings_gdf)

print(f"  Calculated 6 morphological metrics")
print(f"  Average compactness: {buildings_gdf['compactness'].mean():.4f}")
print(f"  Average elongation: {buildings_gdf['elongation'].mean():.4f}")
print(f"  Average convexity: {buildings_gdf['convexity'].mean():.4f}")
# Axis-aligned rectangles: fully convex, no deviation from the cardinal directions
assert np.allclose(buildings_gdf['convexity'], 1)
assert np.allclose(buildings_gdf['orientation'], 0, atol=1e-6)
assert ((buildings_gdf['compactness'] > 0) & (buildings_gdf['compactness'] <= 2 / np.pi + 1e-9)).all()

# Morphological tessellation
print(f"\nTessellation Analysis:")
tess_gdf = momepy.morphological_tessellation(buildings_gdf, shrink=0.4)
tess_gdf = tess_gdf.join(buildings_gdf[['uID', 'area']], how='left')
tess_gdf['tess_area'] = tess_gdf.geometry.area
tess_gdf['coverage'] = buildings_gdf['area'] / tess_gdf['tess_area']

print(f"  Generated {len(tess_gdf)} tessellation cells")
print(f"  Average coverage ratio: {tess_gdf['coverage'].mean():.4f}")
print(f"  Average cell area: {tess_gdf['tess_area'].mean():.0f} m²")
assert len(tess_gdf) == len(buildings_gdf) and (tess_gdf['tess_area'] > 0).all()

print("\n✅ momepy example 5 completed successfully")